# Governed YOLO Training Pipeline

Thin Jupyter/Kaggle interface over `edge_ai_mass.training`. The package owns preprocessing, data lineage, visualization, tuning, training, evaluation, best-model export, and registration; this notebook only supplies environment-specific paths and chooses stages.

Pipeline: **TACO + AquaTrash + RealWaste COCO segmentations → versioned YOLO dataset → Optuna → YOLO → MLflow registry**.

In [ ]:
import json
import os
import sys
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
INPUT_ROOT = Path('/kaggle/input') if IS_KAGGLE else Path('data').resolve()
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path('.').resolve()
print({'is_kaggle': IS_KAGGLE, 'input_root': str(INPUT_ROOT), 'work_root': str(WORK_ROOT)})

## Load the project module

On Kaggle, upload this repository (or a wheel) as a dataset. The cell finds a checkout containing `src/edge_ai_mass`; set `EDGE_AI_MASS_MODULE_ROOT` if auto-discovery is ambiguous.

In [ ]:
def find_module_root() -> Path:
    override = os.getenv('EDGE_AI_MASS_MODULE_ROOT')
    candidates = [Path(override)] if override else []
    candidates += [Path.cwd()]
    if IS_KAGGLE:
        candidates += [path.parent.parent for path in INPUT_ROOT.glob('*/src/edge_ai_mass')]
    for candidate in candidates:
        if candidate and (candidate / 'src' / 'edge_ai_mass').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Upload the project as a Kaggle dataset or set EDGE_AI_MASS_MODULE_ROOT')

MODULE_ROOT = find_module_root()
sys.path.insert(0, str(MODULE_ROOT / 'src'))
CONFIG_PATH = MODULE_ROOT / 'configs' / 'training' / 'yolo_segmentation.yaml'
os.chdir(WORK_ROOT)
print({'module_root': str(MODULE_ROOT), 'config': str(CONFIG_PATH)})

## Bind mounted datasets

Prefer explicit environment variables for stable Kaggle jobs. The local defaults in the YAML already match this repository. RealWaste annotations are the COCO output of `prepare_realwaste_sam_kaggle.ipynb`.

In [ ]:
# Set these before constructing the pipeline when running on Kaggle.
# os.environ['TACO_ANNOTATIONS'] = '/kaggle/input/<taco>/annotations.json'
# os.environ['TACO_IMAGES'] = '/kaggle/input/<taco>'
# os.environ['AQUATRASH_ANNOTATIONS'] = '/kaggle/input/<aquatrash-labels>/labels_final.json'
# os.environ['AQUATRASH_IMAGES'] = '/kaggle/input/<aquatrash>/Images'
# os.environ['REALWASTE_ANNOTATIONS'] = '/kaggle/input/<realwaste-coco>/annotations.json'
# os.environ['REALWASTE_IMAGES'] = '/kaggle/input/<realwaste>/RealWaste'
# os.environ['YOLO_CHECKPOINT'] = '/kaggle/input/<checkpoint>/best.pt'

OVERRIDES = []
if IS_KAGGLE:
    OVERRIDES += [
        'data.output_dir=/kaggle/working/data/processed/waste_seg_yolo',
        'data.materialize=copy',
        'training.artifacts_dir=/kaggle/working/artifacts/training/yolo',
        'tuning.storage=/kaggle/working/artifacts/optuna/yolo.db',
        'tracking.uri=sqlite:////kaggle/working/artifacts/mlflow/mlflow.db',
        'tracking.registry_uri=sqlite:////kaggle/working/artifacts/mlflow/mlflow.db',
    ]
EXPORT_FORMATS = [value.strip() for value in os.getenv('TRAIN_EXPORT_FORMATS', '').split(',') if value.strip()]
if EXPORT_FORMATS:
    OVERRIDES += ['export.enabled=true', f"export.formats=[{', '.join(EXPORT_FORMATS)}]"]
print('Overrides:', OVERRIDES)

In [ ]:
from edge_ai_mass.training import TrainingPipeline

pipeline = TrainingPipeline.from_config(CONFIG_PATH, overrides=OVERRIDES)
plan = pipeline.plan('all')
print(json.dumps(plan, indent=2))

missing = [item['name'] for item in plan['sources'] if item['required'] and not (item['annotations_exist'] and item['images_exist'])]
if missing:
    print('Required mounts still missing:', missing)

## Execute

Use `preprocess` first while validating mounts and dataset mosaics, then resume with `tune,train,evaluate,export,register`. Set `TRAIN_EXPORT_FORMATS=onnx,engine` to request one or multiple best-model exports. Optuna uses persistent storage, and the pipeline state makes stage-by-stage notebook execution resumable.

In [ ]:
RUN_STAGES = os.getenv('TRAIN_STAGES', 'preprocess')
SKIP_OPTUNA = os.getenv('SKIP_OPTUNA', 'false').lower() == 'true'
print({'run_stages': RUN_STAGES, 'skip_optuna': SKIP_OPTUNA})
results = pipeline.run(RUN_STAGES, skip_optuna=SKIP_OPTUNA)
print(json.dumps(results, indent=2, default=str)[:20000])

In [ ]:
# Inspect durable outputs after any stage.
state_path = pipeline.config.artifacts_dir / 'pipeline_state.json'
if state_path.exists():
    print(state_path.read_text())
print('Dataset manifest:', pipeline.config.dataset_dir / 'dataset_manifest.json')
print('Training artifacts:', pipeline.config.artifacts_dir)